# Lesson 7: Prompt Registry

Create the template in Launchpad first — see [docs/07-prompt-registry-launchpad.md](../docs/07-prompt-registry-launchpad.md).

In [ ]:
import sys
sys.path.insert(0, '..')

from src.init_env import set_environment_variables, load_registry_settings
from src.orchestration_helper import get_orchestration_deployment_url
from src.data_loader import load_mails, split_dataset, build_option_lists
from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.prompt_registry import PromptTemplateClient

set_environment_variables()
registry_settings = load_registry_settings()
proxy_client = get_proxy_client(proxy_version='gen-ai-hub')
api_url = get_orchestration_deployment_url(proxy_client.ai_core_client)

mails = load_mails()
dev_set, _, _ = split_dataset(mails)
_, _, _, option_lists = build_option_lists(mails)
mail = dev_set[10]

SCENARIO = registry_settings['PROMPT_SCENARIO']
NAME = registry_settings['PROMPT_NAME']
VERSION = registry_settings['PROMPT_VERSION']
print(f'Looking for template: {SCENARIO}/{NAME}@{VERSION}')

In [ ]:
registry = PromptTemplateClient(proxy_client=proxy_client)

templates = registry.get_prompt_templates(scenario=SCENARIO, name=NAME, version=VERSION)
if not templates.resources:
    raise RuntimeError(
        'Template not found. Create it in AI Launchpad first (see docs/07-prompt-registry-launchpad.md).'
    )

template_resource = templates.resources[0]
print('Template ID:', template_resource.id)
print('Template spec:', template_resource.spec)

In [ ]:
filled = registry.fill_prompt_template_by_id(
    template_id=template_resource.id,
    input_params={
        'input': mail['message'],
        'urgency': option_lists['urgency'],
        'sentiment': option_lists['sentiment'],
        'categories': option_lists['categories'],
    },
)
print('Filled prompt preview:')
print(filled.parsed_prompt[:800])

In [ ]:
from gen_ai_hub.orchestration_v2 import (
    OrchestrationService,
    OrchestrationConfig,
    ModuleConfig,
    PromptTemplatingModuleConfig,
    LLMModelDetails,
    TemplateRefByScenarioNameVersion,
)

config = OrchestrationConfig(
    modules=ModuleConfig(
        prompt_templating=PromptTemplatingModuleConfig(
            prompt=TemplateRefByScenarioNameVersion(
                scenario=SCENARIO,
                name=NAME,
                version=VERSION,
            ),
            model=LLMModelDetails(name='gpt-4o'),
        )
    )
)

orch_v2 = OrchestrationService(api_url=api_url, proxy_client=proxy_client)
result = orch_v2.run(
    config=config,
    placeholder_values={
        'input': mail['message'],
        'urgency': option_lists['urgency'],
        'sentiment': option_lists['sentiment'],
        'categories': option_lists['categories'],
    },
)
print(result.final_result.choices[0].message.content)